In [20]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [8]:
load_dotenv()

True

In [5]:
class EvaluationSchema(BaseModel):
    feedback:str= Field(description="Detailed feedback for the essay")
    score: int= Field(description="Score out of 10 for the essay",ge=0, le=10)
    

In [10]:
model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0)
structured_model = model.with_structured_output(
    EvaluationSchema, 
    method="function_calling" # Forces stable schema translation
)

In [14]:
structured_model=model.with_structured_output(EvaluationSchema)


In [12]:
essay="""
When Neon Genesis Evangelion debuted in 1995, it completely shattered the traditional "giant robot" mecha genre. Instead of focusing on simple action, creator Hideaki Anno injected his own real-world struggles with clinical depression into the narrative. The story traded classic heroic tropes for a raw, agonizing look at teenage trauma, isolation, and anxiety.
Today, Evangelion’s cinematic footprint is everywhere. You can see its direct influence in the scale and biomechanical concepts of Guillermo del Toro’s Pacific Rim (2013), as well as the deep, identity-driven existential dread found in the sci-fi blockbusters of Christopher Nolan and Denis Villeneuve. It permanently expanded the boundaries of what genre filmmaking could achieve.
"""

In [13]:
prompt = f"""
Evaluate the language quality (grammar, vocabulary, flow) of the following essay. 
Ignore content accuracy. Score 0 to 10 (9-10=flawless, 7-8=minor errors, 5-6=weak flow, 0-4=broken syntax).

[ESSAY]
{essay}
"""

In [ ]:
structured_model.invoke(prompt)

'The essay is well-written with flawless grammar, vocabulary, and flow. The sentences are structured logically, and the language is sophisticated. The writer effectively uses transitional phrases and rhetorical devices to convey their ideas. The tone is formal and engaging, making it easy to follow the argument. Overall, the language quality is exceptional.'

In [22]:
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add] # reducer function (add)
    avg_score: float


In [28]:
def evaluate_language(state:EssayState):

    prompt=f"""
Evaluate the language quality (grammar, vocabulary, flow) of the following essay. 
Ignore content accuracy. Score 0 to 10 (9-10=flawless, 7-8=minor errors, 5-6=weak flow, 0-4=broken syntax).

[ESSAY]
{state['essay']}
 
"""
    output=structured_model.invoke(prompt)
    return  {'language_feedback': output.feedback, 'individual_scores':[output.score]}


In [29]:
def evaluate_analysis(state:EssayState):

    prompt=f"""
Evaluate the depth of analysis of the following essay. 
Ignore content accuracy. Score 0 to 10 (9-10=flawless, 7-8=minor errors, 5-6=weak flow, 0-4=broken syntax).

[ESSAY]
{state['essay']}
 
"""
    output=structured_model.invoke(prompt)
    return  {'analysis_feedback': output.feedback, 'individual_scores':[output.score]}


In [30]:
def evaluate_thought(state:EssayState):

    prompt=f"""
Evaluate the clarityn of thought of the following essay. 
Ignore content accuracy. Score 0 to 10 (9-10=flawless, 7-8=minor errors, 5-6=weak flow, 0-4=broken syntax).

[ESSAY]
{state['essay']}
 
"""
    output=structured_model.invoke(prompt)
    return  {'clarity_feedback': output.feedback, 'individual_scores':[output.score]}


In [31]:
def final_evaluation  (state:EssayState):

    prompt=f"""
Based on the following feedbacks create a summarized feedback \n language feedback - {state['language_feedback']} \n analysis feedback - {state['analysis_feedback']} \n clarity feedback - {state['clarity_feedback']} \n Also provide an overall score out of 10 for the essay based on the average of the individual scores.
 
"""
    overall_feedback=model.invoke(prompt).content
    
    # avg calc
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {
        'overall_feedback': overall_feedback,
        'avg_score': avg_score  
    }


In [37]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()


In [39]:
initial_state = {
    'essay': essay
}

workflow.invoke(initial_state)

{'essay': '\nWhen Neon Genesis Evangelion debuted in 1995, it completely shattered the traditional "giant robot" mecha genre. Instead of focusing on simple action, creator Hideaki Anno injected his own real-world struggles with clinical depression into the narrative. The story traded classic heroic tropes for a raw, agonizing look at teenage trauma, isolation, and anxiety.\nToday, Evangelion’s cinematic footprint is everywhere. You can see its direct influence in the scale and biomechanical concepts of Guillermo del Toro’s Pacific Rim (2013), as well as the deep, identity-driven existential dread found in the sci-fi blockbusters of Christopher Nolan and Denis Villeneuve. It permanently expanded the boundaries of what genre filmmaking could achieve.\n',
 'language_feedback': "The essay is well-written with flawless grammar, vocabulary, and flow. The sentences are complex and varied, with a clear and logical structure. The language is engaging and expressive, effectively conveying the au

In [40]:
shitty_essay = """
Evangelion is like about big robots in 1995 which is old but it shattered mecha stuff completely. Because the maker guy Hideaki Anno have very bad clinical depression so he putted his sad brains into the show. Its not heroic it just have teenagers crying and trauma and anxiety which is crazy.
Now days its footprint is everywheres. Like Pacific Rim by that toro guy in 2013 have the same big robot concepts and biomechanics thingies. Also Christopher nolan and denis villeneuve movies have the same existential dread stuff where no one knows who they are. It make genre filmmaking bigger forever i think.
"""

# Test the workflow with the broken text
initial_state = {
    'essay': shitty_essay
}

# Invoke your compiled LangGraph pipeline
output = workflow.invoke(initial_state)
print(output)

{'essay': '\nEvangelion is like about big robots in 1995 which is old but it shattered mecha stuff completely. Because the maker guy Hideaki Anno have very bad clinical depression so he putted his sad brains into the show. Its not heroic it just have teenagers crying and trauma and anxiety which is crazy.\nNow days its footprint is everywheres. Like Pacific Rim by that toro guy in 2013 have the same big robot concepts and biomechanics thingies. Also Christopher nolan and denis villeneuve movies have the same existential dread stuff where no one knows who they are. It make genre filmmaking bigger forever i think.\n', 'language_feedback': "The essay has several grammatical errors, such as 'which is old but it shattered mecha stuff completely', 'he putted his sad brains into the show', and 'Its not heroic'. The vocabulary is simple and lacks precision, with phrases like 'big robots', 'sad brains', and 'existential dread stuff'. The flow of the essay is weak, with abrupt transitions betwee